# 🌍 Introduction to Geospatial Libraries — Exercise Notebook

---

> **How to use this notebook:**  
> Each section contains **worked examples** you can run, followed by **practice exercises**.  
> Read the examples carefully, then attempt the exercises in the solution cells.  
> Use `Shift + Enter` to run a cell.

---

## 📋 Table of Contents

| # | Library | Purpose |
|---|---------|----------|
| 0 | Setup & Environment | Google Colab, packages, data download |
| 1 | **Shapely** | Vector Geometry |
| 2 | **Fiona** | Vector I/O |
| 3 | **GeoPandas** | Vector Analysis |
| 4 | **Rasterio** | Raster I/O |
| 5 | **Xarray** | N-D Array Handling |
| 6 | **Rioxarray** | Raster/N-D Bridge |
| 7 | **scipy.stats** | Statistical Analysis |
| 8 | **Folium** | Interactive Mapping |
| 9 | **Leafmap** | Rapid Visualization |
| 10 | **Google Earth Engine (ee)** | Cloud GIS API |
| 11 | **rasterstats** | Zonal Statistics |
| 12 | **Matplotlib** | Static Visualization |
| 13 | **Plotly** | Interactive Visualization |

---

# 0️⃣ Setup & Environment

---

> **Google Colab** is a hosted Jupyter notebook environment. No local installation is required.  
> Every session runs on a fresh Ubuntu Linux machine in the cloud.

| Feature | Google Colab | Local Jupyter |
|---------|:---:|:---:|
| Installation needed | ❌ | ✅ |
| Free GPU/TPU | ✅ | ❌ |
| Internet required | ✅ | ❌ |
| Easy sharing | ✅ | ⚠️ |

---

### 💡 Example 0.1 — Hello World & Python Version

In [ ]:
import sys
print('Hello World')
print('Python version:', sys.version)
env = 'Google Colab' if 'google.colab' in sys.modules else 'Local Jupyter'
print('Running in:', env)

### 💡 Example 0.2 — Package Management

In [ ]:
import pandas as pd
import geopandas as gpd

# Install a package if needed (uncomment):
# !pip install rioxarray
# import rioxarray

# List all installed packages:
# !pip list -v

### 💡 Example 0.3 — Data & Output Folders

In [ ]:
import os

data_folder   = 'data'
output_folder = 'output'

for folder in [data_folder, output_folder]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print(f'Created folder: {folder}')
    else:
        print(f'Folder already exists: {folder}')

### 💡 Example 0.4 — Download Helper & Sample Data

In [ ]:
import requests

def download(url):
    """Download a file from url into the data folder."""
    filename = os.path.join(data_folder, os.path.basename(url))
    if not os.path.exists(filename):
        with requests.get(url, stream=True, allow_redirects=True) as r:
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print('Downloaded', filename)
    else:
        print('Already exists:', filename)

# Download tract-level shapefiles for three US states
base_url   = 'https://www2.census.gov/geo/tiger/TIGER2025/TRACT/'
zones_files = ['tl_2025_38_tract.zip', 'tl_2025_39_tract.zip', 'tl_2025_40_tract.zip']

for z in zones_files:
    download(base_url + z)

# 1️⃣ Vector Geometry: Shapely

---

> **Shapely** creates and manipulates planar geometric objects (Points, Lines, Polygons) entirely in memory — no file I/O required.  
> It is the building block for almost every other Python geospatial library.

| Geometry Type | Constructor |
|---|---|
| Point | `Point(x, y)` |
| LineString | `LineString([(x1,y1), (x2,y2), ...])` |
| Polygon | `Polygon(shell, [holes])` |
| MultiPoint | `MultiPoint([...])` |
| MultiLineString | `MultiLineString([...])` |
| MultiPolygon | `MultiPolygon([...])` |
| GeometryCollection | `GeometryCollection([...])` |

---

### 💡 Example 1.1 — The Seven Geometry Types

In [ ]:
from shapely.geometry import (
    Point, LineString, Polygon,
    MultiPoint, MultiLineString, MultiPolygon,
    GeometryCollection
)

pt   = Point(5, 2)
ls   = LineString([(1,5),(4,4),(4,1),(2,2),(3,2)])
poly = Polygon([(1,5),(2,2),(4,1),(4,4),(1,5)])
mp   = MultiPoint([(5,2),(1,3),(3,4),(3,2)])
mls  = MultiLineString([[(1,5),(4,4),(4,1)],[(1,2),(2,4)]])
outer = [(1,5),(2,2),(4,1),(4,4),(1,5)]
hole  = [(0,2),(1,2),(1,3),(0,3),(0,2)]
mpoly = MultiPolygon([Polygon(outer, [hole])])
gc    = GeometryCollection([mp, ls])

for name, geom in [('POINT',pt),('LINESTRING',ls),('POLYGON',poly),
                   ('MULTIPOINT',mp),('MULTILINESTRING',mls),
                   ('MULTIPOLYGON',mpoly),('GEOMETRYCOLLECTION',gc)]:
    print(f'{name}: {geom.wkt[:60]}')
    display(geom)

### 💡 Example 1.2 — Spatial Predicates & Measurements

In [ ]:
from shapely.geometry import Point, LineString, Polygon
from shapely import wkt

# Buffer + intersection
pt     = Point(5, 5)
buffer = pt.buffer(3)
line   = LineString([(8,5),(12,5)])
print('Buffer intersects line:', buffer.intersects(line))
display(buffer)

# Polygon area and centroid
outer = [(0,0),(6,0),(6,6),(0,6),(0,0)]
hole  = [(2,2),(4,2),(4,4),(2,4),(2,2)]
poly  = Polygon(outer, [hole])
print('Area with hole:', poly.area)
print('Centroid:', poly.centroid)
display(poly)

# WKT round-trip
wkt_str  = poly.wkt
restored = wkt.loads(wkt_str)
print('WKT restored:', restored.geom_type)

### 💡 Example 1.3 — Convex Hull & Geometry Filtering

In [ ]:
from shapely.geometry import MultiPoint, GeometryCollection, Point, LineString

# Convex hull of a MultiPoint
mp   = MultiPoint([(1,1),(5,1),(3,4),(2,2),(4,3)])
hull = mp.convex_hull
print('Convex Hull:', hull.wkt)
display(GeometryCollection([mp, hull]))

# Extract only Points from a GeometryCollection
gc     = GeometryCollection([Point(1,1), LineString([(0,0),(2,2)]), Point(5,5), Point(3,4)])
points = [g for g in gc.geoms if g.geom_type == 'Point']
print(f'Extracted {len(points)} points from collection')
for p in points:
    print(' ', p)

# 2️⃣ Vector I/O: Fiona

---

> **Fiona** reads and writes geospatial vector formats (Shapefiles, GeoJSON, GeoPackage) through a clean Pythonic interface to GDAL.  
> Every feature is returned as a Python dictionary — easy to inspect and transform.

| Task | Fiona method |
|------|--------------|
| Open for reading | `fiona.open(path, 'r')` |
| Get CRS | `src.crs` |
| Iterate features | `for feat in src:` |
| Open for writing | `fiona.open(path, 'w', driver=..., crs=..., schema=...)` |
| Write feature | `dst.write({...})` |

---

### 💡 Example 2.1 — Install & Read a Shapefile

In [ ]:
# !pip install fiona   # uncomment if needed
import fiona

file = r'data/tl_2025_38_tract.shp'

with fiona.open(file, 'r') as src:
    print('CRS       :', src.crs)
    print('Driver    :', src.driver)
    print('Schema    :', src.schema)
    print('Feature count:', len(src))
    # Print first feature
    first = next(iter(src))
    print('\nFirst geometry type:', first['geometry']['type'])
    print('First properties   :', dict(first['properties']))

### 💡 Example 2.2 — Write a New Shapefile

In [ ]:
import fiona
from fiona.crs import from_epsg

schema = {
    'geometry'  : 'Point',
    'properties': {'name': 'str:20', 'value': 'float'}
}

out_path = 'output/sample_points.shp'

with fiona.open(out_path, 'w', driver='ESRI Shapefile',
                crs=from_epsg(4326), schema=schema) as dst:
    dst.write({
        'geometry'  : {'type': 'Point', 'coordinates': (10, 20)},
        'properties': {'name': 'SiteA', 'value': 3.14}
    })
    dst.write({
        'geometry'  : {'type': 'Point', 'coordinates': (85.3, 27.7)},
        'properties': {'name': 'Kathmandu', 'value': 1400.0}
    })

print('Written:', out_path)

# 3️⃣ Vector Analysis: GeoPandas

---

> **GeoPandas** extends Pandas DataFrames with a `geometry` column, enabling vectorized spatial operations directly on tabular data.

| Operation | Method |
|-----------|--------|
| Read file | `gpd.read_file(path)` |
| Calculate area | `gdf.area` |
| Reproject | `gdf.to_crs(epsg=...)` |
| Spatial join | `gpd.sjoin(left, right, predicate=...)` |
| Dissolve | `gdf.dissolve(by='column')` |
| Buffer | `gdf.buffer(distance)` |
| Write file | `gdf.to_file(path)` |

---

### 💡 Example 3.1 — Create & Inspect a GeoDataFrame

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon

data = {
    'city'      : ['Fargo', 'Bismarck', 'Grand Forks'],
    'population': [160000, 43000, 59000],
    'geometry'  : [
        Polygon([(2,5),(7,9),(6,8),(5,1)]),
        Polygon([(10,10),(10,12),(12,12),(12,10)]),
        Polygon([(0,0),(3,0),(3,3),(0,3)])
    ]
}
gdf = gpd.GeoDataFrame(data, crs='EPSG:4326')
gdf['area'] = gdf.area
print(gdf[['city','population','area']])
gdf.plot(column='population', legend=True, figsize=(6,4))

### 💡 Example 3.2 — Read Shapefile, Filter & Reproject

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

nd    = gpd.read_file(r'data/tl_2025_38_tract.zip')
print('CRS:', nd.crs)
print('Shape:', nd.shape)

# Filter to a single county
county = nd[nd['COUNTYFP'] == '017']
print(f'\nCounty 017 tracts: {len(county)}')

# Reproject to a projected CRS for accurate area calculation
county_proj = county.to_crs(epsg=32614)
county_proj['area_km2'] = county_proj.area / 1e6
print(county_proj[['GEOID','area_km2']].head())

county.plot(color='lightyellow', edgecolor='black', figsize=(6,5))
plt.title('County 017 Tracts')
plt.show()

### 💡 Example 3.3 — Spatial Join

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Polygons (zones)
zones = gpd.GeoDataFrame({
    'zone': ['A','B'],
    'geometry': [
        Polygon([(0,0),(5,0),(5,5),(0,5)]),
        Polygon([(5,0),(10,0),(10,5),(5,5)])
    ]
}, crs='EPSG:4326')

# Points (locations)
pts = gpd.GeoDataFrame({
    'name': ['P1','P2','P3'],
    'geometry': [Point(2,2), Point(7,3), Point(4,4)]
}, crs='EPSG:4326')

joined = gpd.sjoin(pts, zones, how='left', predicate='within')
print(joined[['name','zone']])

# 4️⃣ Raster I/O: Rasterio

---

> **Rasterio** reads and writes GeoTIFFs and other raster formats, exposing data as NumPy arrays with full geospatial metadata.

| Task | Code |
|------|------|
| Open raster | `rasterio.open(path)` |
| Read band | `src.read(1)` |
| CRS & transform | `src.crs`, `src.transform` |
| Clip to geometry | `rasterio.mask.mask(src, shapes)` |
| Write raster | `rasterio.open(path,'w', **profile)` |

---

### 💡 Example 4.1 — Read Raster Metadata

In [ ]:
# !pip install rasterio   # uncomment if needed
import rasterio
import numpy as np

# --- Using a real file: uncomment and replace path ---
# with rasterio.open('data/my_dem.tif') as src:
#     print('CRS     :', src.crs)
#     print('Bounds  :', src.bounds)
#     print('Shape   :', src.height, src.width)
#     print('Bands   :', src.count)
#     band1 = src.read(1)
#     print('Min/Max :', band1.min(), band1.max())

# Mock demo (no file required)
class MockRaster:
    crs     = 'EPSG:4326'
    bounds  = (10.0, 20.0, 30.0, 40.0)
    count   = 4
    height  = 512
    width   = 512
    def read(self, band): return np.random.randint(0,255,(self.height,self.width))

src = MockRaster()
print(f'CRS   : {src.crs}')
print(f'Bands : {src.count}')
print(f'Size  : {src.height} x {src.width}')
band = src.read(1)
print(f'Mean pixel value: {band.mean():.2f}')

### 💡 Example 4.2 — Clipping a Raster with a Vector Mask

In [ ]:
# Conceptual example — requires real raster file
# from rasterio.mask import mask as rio_mask
# import geopandas as gpd
#
# gdf  = gpd.read_file('data/my_boundary.shp').to_crs('EPSG:4326')
# shapes = [geom.__geo_interface__ for geom in gdf.geometry]
#
# with rasterio.open('data/my_dem.tif') as src:
#     clipped, clipped_transform = rio_mask(src, shapes, crop=True)
#     meta = src.meta.copy()
#
# meta.update({'height': clipped.shape[1], 'width': clipped.shape[2],
#              'transform': clipped_transform})
#
# with rasterio.open('output/clipped.tif', 'w', **meta) as dst:
#     dst.write(clipped)
#
print('Clipping concept: rasterio.mask.mask(src, shapes, crop=True)')
print('Result is a NumPy array + new affine transform.')

# 5️⃣ N-D Array Handling: Xarray

---

> **Xarray** adds named dimensions and coordinates to NumPy arrays — ideal for multi-dimensional data like climate grids and time series.

| Concept | Description |
|---------|-------------|
| `DataArray` | Labelled N-D array |
| `Dataset` | Dict of DataArrays sharing coordinates |
| `.sel()` | Select by label value |
| `.isel()` | Select by integer index |
| `.mean(dim=...)` | Reduce along a dimension |

---

### 💡 Example 5.1 — Create & Select from a DataArray

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

times = pd.date_range('2023-01-01', periods=5)
lats  = np.arange(40, 45)

temp  = xr.DataArray(
    np.random.rand(5, 5) * 30,
    coords=[times, lats],
    dims=['time', 'lat'],
    name='temperature'
)

print(temp)
print('\nAt lat=42:', temp.sel(lat=42).values)
print('Mean over time:', temp.mean(dim='time').values)

### 💡 Example 5.2 — Dataset & Arithmetic

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

times = pd.date_range('2023-01-01', periods=5)
lats  = np.arange(40, 45)

ds = xr.Dataset({
    'temperature': xr.DataArray(np.random.rand(5,5)*30, dims=['time','lat'],
                                coords={'time':times,'lat':lats}),
    'humidity'   : xr.DataArray(np.random.rand(5,5)*100, dims=['time','lat'],
                                coords={'time':times,'lat':lats})
})

print(ds)
print('\nMean temperature:', ds['temperature'].mean().item(), '°C')
print('Max humidity    :', ds['humidity'].max().item(), '%')

# 6️⃣ Raster/N-D Bridge: Rioxarray

---

> **Rioxarray** extends Xarray with a `.rio` accessor that exposes CRS, spatial transforms, reprojection, and clipping — bridging Rasterio and Xarray seamlessly.

| `.rio` method | Purpose |
|---------------|---------|
| `.crs` | Get the CRS |
| `.write_crs(epsg)` | Set / write the CRS |
| `.reproject('EPSG:...')` | Reproject |
| `.clip(geometries, crs)` | Clip to vector boundary |
| `.to_raster(path)` | Save to GeoTIFF |

---

### 💡 Example 6.1 — Access CRS via .rio Accessor

In [ ]:
# !pip install rioxarray   # uncomment if needed
import xarray as xr
import numpy as np
import rioxarray   # registers the .rio accessor

# Create a mock spatial DataArray
da = xr.DataArray(
    np.random.rand(4, 4),
    dims=('y', 'x'),
    coords={'y': [40.5, 41.5, 42.5, 43.5], 'x': [-97.5, -96.5, -95.5, -94.5]}
)
da = da.rio.write_crs('EPSG:4326')
print('CRS  :', da.rio.crs)
print('Shape:', da.rio.shape)
print('Bounds:', da.rio.bounds())

# 7️⃣ Statistical Analysis: scipy.stats

---

> **scipy.stats** provides probability distributions, hypothesis tests, and descriptive statistics critical for spatial data analysis.

| Function | Purpose |
|----------|---------|
| `stats.describe(data)` | Summary statistics |
| `stats.ttest_1samp(data, popmean)` | One-sample t-test |
| `stats.pearsonr(x, y)` | Pearson correlation |
| `stats.norm.fit(data)` | Fit normal distribution |
| `stats.kstest(data, 'norm')` | Normality test |

---

### 💡 Example 7.1 — Descriptive Stats & Hypothesis Test

In [ ]:
from scipy import stats
import numpy as np

# Simulated elevation data (metres)
elevation = np.array([125.5, 130.2, 120.8, 145.0, 118.9, 132.1, 128.7, 140.3])

desc = stats.describe(elevation)
print(f'N     : {desc.nobs}')
print(f'Mean  : {desc.mean:.2f} m')
print(f'Std   : {np.sqrt(desc.variance):.2f} m')
print(f'Min   : {desc.minmax[0]:.2f} m')
print(f'Max   : {desc.minmax[1]:.2f} m')

# One-sample t-test: is our mean significantly different from 135 m?
t_stat, p_val = stats.ttest_1samp(elevation, 135.0)
print(f'\nt-statistic : {t_stat:.4f}')
print(f'p-value     : {p_val:.4f}')
print('Significant difference (p<0.05):', p_val < 0.05)

### 💡 Example 7.2 — Correlation & Distribution Fitting

In [ ]:
from scipy import stats
import numpy as np

# Simulate NDVI and rainfall
np.random.seed(42)
rainfall = np.random.normal(500, 80, 30)
ndvi     = 0.003 * rainfall + np.random.normal(0, 0.05, 30)

r, p = stats.pearsonr(rainfall, ndvi)
print(f'Pearson r = {r:.4f},  p = {p:.4f}')

# Fit a normal distribution to rainfall
mu, sigma = stats.norm.fit(rainfall)
print(f'\nFitted Normal: mean={mu:.1f}, std={sigma:.1f}')

# 8️⃣ Interactive Mapping: Folium

---

> **Folium** generates interactive Leaflet.js maps in Python notebooks and HTML files — no JavaScript required.

| Element | Code |
|---------|------|
| Base map | `folium.Map(location, zoom_start, tiles)` |
| Marker | `folium.Marker(location, popup, tooltip)` |
| Circle | `folium.Circle(location, radius, color)` |
| GeoJSON layer | `folium.GeoJson(data).add_to(m)` |
| Save map | `m.save('map.html')` |

---

### 💡 Example 8.1 — Basic Folium Map with Markers

In [ ]:
import folium

fargo_coords = (46.8772, -96.7898)

m = folium.Map(location=fargo_coords, zoom_start=10, tiles='OpenStreetMap')

folium.Marker(fargo_coords,
              popup='<b>Fargo, ND</b>',
              tooltip='Click me').add_to(m)

folium.Circle(fargo_coords, radius=5000,
              color='blue', fill=True, fill_opacity=0.2,
              tooltip='5 km radius').add_to(m)

# Display in notebook
m

### 💡 Example 8.2 — Multiple Markers & Save to HTML

In [ ]:
import folium

cities = [
    ('Fargo',       46.8772, -96.7898, 'blue'),
    ('Bismarck',    46.8083, -100.7837, 'red'),
    ('Grand Forks', 47.9253, -97.0329,  'green'),
]

m = folium.Map(location=[47.0, -99.0], zoom_start=6)

for name, lat, lon, color in cities:
    folium.Marker(
        [lat, lon],
        popup=name,
        icon=folium.Icon(color=color)
    ).add_to(m)

m.save('output/nd_cities_map.html')
print('Map saved to output/nd_cities_map.html')
m

# 9️⃣ Rapid Visualization: Leafmap

---

> **Leafmap** simplifies interactive geospatial visualization — add basemaps, vector layers, and rasters in a few lines.

| Task | Method |
|------|--------|
| Create map | `leafmap.Map(center, zoom)` |
| Add basemap | `m.add_basemap('OpenStreetMap')` |
| Add GeoDataFrame | `m.add_gdf(gdf, layer_name)` |
| Add GeoJSON URL | `m.add_geojson(url, layer_name)` |
| Save to HTML | `m.to_html(path)` |
| Zoom to data | `m.zoom_to_gdf(gdf)` |

---

### 💡 Example 9.1 — Leafmap Map with Vector Layer

In [ ]:
# !pip install leafmap   # uncomment if needed
import leafmap
import geopandas as gpd

m = leafmap.Map(center=(46.88, -99.0), zoom=6)
m.add_basemap('OpenStreetMap')

# Load and display ND tract data
gdf = gpd.read_file(r'data/tl_2025_38_tract.zip')
m.add_gdf(gdf, layer_name='ND Tracts',
          style={'color': 'navy', 'weight': 0.5, 'fillOpacity': 0.2})
m.zoom_to_gdf(gdf)

m

### 💡 Example 9.2 — Save to HTML

In [ ]:
output_path = r'output/nd_tracts_leafmap.html'
m.to_html(output_path)
print('Saved to:', output_path)

# 🔟 Cloud GIS API: Google Earth Engine (ee)

---

> **Google Earth Engine** enables petabyte-scale geospatial analysis on Google's cloud infrastructure.  
> Computation is **server-side** — only results are returned to your local machine.

| Step | Code |
|------|------|
| Authenticate | `ee.Authenticate()` |
| Initialize | `ee.Initialize(project='your-project')` |
| Load collection | `ee.ImageCollection('LANDSAT/...')` |
| Filter dates | `.filterDate('YYYY-MM-DD', 'YYYY-MM-DD')` |
| Compute | `.median()`, `.mean()`, `.mosaic()` |
| Get value | `.getInfo()` |

---

### 💡 Example 10.1 — Authenticate, Initialize & Load Landsat 8

In [ ]:
import ee

# Authenticate (only needed once per session)
ee.Authenticate()
ee.Initialize(project='rsclass01')  # replace with your project ID

# Load Landsat 8 TOA collection and filter by date
collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2024-01-01', '2024-06-30')
)

print('Images in collection:', collection.size().getInfo())

# Compute median composite
median_img = collection.median()
print('Median image band names:', median_img.bandNames().getInfo())

# 1️⃣1️⃣ Zonal Statistics: rasterstats

---

> **rasterstats** calculates summary statistics (mean, max, sum, …) of raster pixels that fall within vector boundaries — the core of zonal statistics.

| Parameter | Description |
|-----------|-------------|
| `vector` | Path or GeoDataFrame of zones |
| `raster` | Path or NumPy array |
| `stats` | List: `['mean','max','min','count','sum']` |
| `affine` | Affine transform (when passing array) |
| Returns | List of dicts, one per feature |

---

### 💡 Example 11.1 — Zonal Statistics on a Mock Raster

In [ ]:
# !pip install rasterstats   # uncomment if needed
import numpy as np
from shapely.geometry import Polygon
from rasterstats import zonal_stats
from affine import Affine

# 5×5 mock raster
raster = np.array([
    [10, 20, 30, 40, 50],
    [15, 25, 35, 45, 55],
    [12, 22, 32, 42, 52],
    [18, 28, 38, 48, 58],
    [11, 21, 31, 41, 51]
], dtype=float)

# Affine(xres, xskew, xorigin, yskew, yres, yorigin)
affine = Affine(1.0, 0.0, 0.0,
                0.0,-1.0, 5.0)

# Define two zones
zone1 = Polygon([(0,0),(0,3),(2,3),(2,0)])
zone2 = Polygon([(2,0),(2,3),(5,3),(5,0)])

for i, zone in enumerate([zone1, zone2], 1):
    result = zonal_stats(zone, raster, affine=affine,
                         stats=['mean','max','min','sum','count'],
                         nodata=-9999)
    print(f'Zone {i}: {result[0]}')

# 1️⃣2️⃣ Static Visualization: Matplotlib

---

> **Matplotlib** is Python's foundational plotting library — highly customisable for maps, charts, and multi-panel figures.

| Plot type | Function |
|-----------|----------|
| Line chart | `plt.plot(x, y)` |
| Histogram | `plt.hist(data)` |
| Scatter | `plt.scatter(x, y)` |
| Geo plot | `gdf.plot(ax=ax, column=..., cmap=...)` |
| Save figure | `plt.savefig('file.png', dpi=150)` |

---

### 💡 Example 12.1 — Basic Line & Scatter Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Line plot
ax1.plot(x, np.cos(x), label='cos(x)', color='blue')
ax1.plot(x, np.sin(x), label='sin(x)', color='orange')
ax1.set_title('Trig Functions'); ax1.legend(); ax1.grid(True)

# Scatter plot
np.random.seed(0)
ax2.scatter(np.random.rand(50)*10, np.random.rand(50)*10,
            c=np.random.rand(50), cmap='viridis', s=80)
ax2.set_title('Random Scatter')

plt.tight_layout()
plt.show()

### 💡 Example 12.2 — Choropleth Map of ND Tracts

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

nd      = gpd.read_file(r'data/tl_2025_38_tract.zip')
nd_proj = nd.to_crs(epsg=32614)
nd_proj['area_km2'] = nd_proj.area / 1e6

fig, ax = plt.subplots(figsize=(10, 8))
nd_proj.plot(ax=ax, column='area_km2', cmap='YlOrRd',
             legend=True, legend_kwds={'label': 'Area (km²)'})
ax.set_title('North Dakota Census Tracts — Area', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('output/nd_choropleth.png', dpi=150)
plt.show()
print('Saved: output/nd_choropleth.png')

# 1️⃣3️⃣ Interactive Visualization: Plotly

---

> **Plotly** creates publication-quality interactive charts — hover, zoom, and filter in the browser.

| Chart type | Function |
|------------|----------|
| Line | `px.line(df, x=, y=)` |
| Scatter | `px.scatter(df, x=, y=, color=)` |
| Bar | `px.bar(df, x=, y=)` |
| Choropleth | `px.choropleth(df, geojson=, locations=, color=)` |
| 3D scatter | `px.scatter_3d(df, x=, y=, z=, color=)` |

---

### 💡 Example 13.1 — Interactive Scatter Plot

In [ ]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    'Year'  : [2000, 2005, 2010, 2015, 2020, 2000, 2005, 2010, 2015, 2020],
    'NDVI'  : [0.55, 0.60, 0.58, 0.65, 0.70, 0.40, 0.42, 0.38, 0.45, 0.50],
    'Region': ['Forest']*5 + ['Grassland']*5,
    'Area'  : [100, 110, 105, 120, 130, 80, 82, 79, 88, 95]
})

fig = px.scatter(
    df, x='Year', y='NDVI', color='Region', size='Area',
    title='NDVI Trend by Land Cover Region',
    hover_name='Region'
)
fig.show()

### 💡 Example 13.2 — Interactive Line & Bar Charts

In [ ]:
import plotly.express as px
import pandas as pd

# Line chart — temperature trend
df_temp = pd.DataFrame({
    'Month': ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
    'Temp_C': [-13, -10, -3, 7, 15, 21, 24, 23, 16, 7, -3, -11]
})
fig_line = px.line(df_temp, x='Month', y='Temp_C',
                   title='Average Monthly Temperature — Fargo, ND',
                   markers=True)
fig_line.show()

# Bar chart — population
df_pop = pd.DataFrame({
    'City': ['Fargo','Bismarck','Grand Forks','Minot','Dickinson'],
    'Pop' : [125000, 74000, 59000, 48000, 25000]
})
fig_bar = px.bar(df_pop, x='City', y='Pop',
                 title='North Dakota City Populations',
                 color='City')
fig_bar.show()

---

# 🎉 Congratulations — All 14 Sections Complete!

---

| ✅ | Section | Library |
|----|---------|----------|
| ✅ | 0 | Setup & Environment |
| ✅ | 1 | Shapely — Vector Geometry |
| ✅ | 2 | Fiona — Vector I/O |
| ✅ | 3 | GeoPandas — Vector Analysis |
| ✅ | 4 | Rasterio — Raster I/O |
| ✅ | 5 | Xarray — N-D Arrays |
| ✅ | 6 | Rioxarray — Raster/N-D Bridge |
| ✅ | 7 | scipy.stats — Statistics |
| ✅ | 8 | Folium — Interactive Maps |
| ✅ | 9 | Leafmap — Rapid Visualization |
| ✅ | 10 | Google Earth Engine |
| ✅ | 11 | rasterstats — Zonal Statistics |
| ✅ | 12 | Matplotlib — Static Visualization |
| ✅ | 13 | Plotly — Interactive Visualization |

---

## 🚀 Next Steps

| Domain | Libraries |
|--------|-----------|
| 🛰️ Remote Sensing | `rasterio`, `rioxarray`, `earthpy`, `ee` |
| 🗺️ Web Mapping | `folium`, `leafmap`, `ipyleaflet`, `kepler.gl` |
| 📊 Spatial Stats | `pysal`, `esda`, `libpysal` |
| ☁️ Cloud GIS | `geemap`, `planetary-computer`, `stac` |
| 🔧 Automation | `snakemake`, `prefect`, `airflow` |

---

### Happy Mapping! 🌍💻

*Every dataset has a story — geospatial libraries help you read it.*

---